# Modelo-07 — Treinamento do CatBoostClassifier

## Objetivo

Este notebook treina e avalia um `CatBoostClassifier` para prever se um sistema PSD apresentará ao menos uma falha confirmada entre `D+1` e `D+7`.

O treinamento utilizará as bases temporais produzidas no notebook de processamento:

- **treino:** ajuste dos parâmetros internos do modelo;
- **validação:** acompanhamento da generalização, seleção de iterações e ajuste do limiar de decisão;
- **teste:** avaliação final, utilizada somente depois que o modelo e o limiar estiverem definidos.

As principais métricas serão `PR-AUC`, `ROC-AUC`, `recall`, `precision`, `F1` e matriz de confusão. Como o objetivo é antecipar falhas, o limiar não será escolhido apenas pela acurácia: será necessário equilibrar falhas detectadas e quantidade de alertas falsos.

> O conjunto de teste não será utilizado para ajustar hiperparâmetros, selecionar features ou escolher o limiar de classificação.

In [3]:
from pathlib import Path
import json

import catboost
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from catboost import CatBoostClassifier, Pool
from IPython.display import display
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda valor: f"{valor:,.4f}")

print(f"CatBoost: {catboost.__version__}")
print(f"Pandas: {pd.__version__}")
print(f"Scikit-learn: {sklearn.__version__}")

CatBoost: 1.2.10
Pandas: 3.0.5
Scikit-learn: 1.9.0


In [5]:
DIRETORIO_DADOS = Path("../../data/processados/modelo-07")

with (DIRETORIO_DADOS / "configuracao_modelagem.json").open(encoding="utf-8") as arquivo:
    configuracao = json.load(arquivo)

TARGET = configuracao["target"]
features_categoricas = configuracao["features_categoricas"]
features_numericas = configuracao["features_numericas"]
features_modelo = configuracao["features_modelo"]
RANDOM_SEED = configuracao["random_seed"]

def carregar_base(nome_conjunto):
    df = pd.read_csv(
        DIRETORIO_DADOS / f"base_{nome_conjunto}.csv",
        parse_dates=["data_referencia"],
    )

    for coluna in features_categoricas:
        df[coluna] = df[coluna].astype(str)

    return df

base_treino = carregar_base("treino")
base_validacao = carregar_base("validacao")
base_teste = carregar_base("teste")

auditoria_bases = pd.DataFrame([
    {
        "conjunto": nome,
        "linhas": len(df),
        "data_inicial": df["data_referencia"].min(),
        "data_final": df["data_referencia"].max(),
        "taxa_positivos": df[TARGET].mean() * 100,
    }
    for nome, df in {
        "treino": base_treino,
        "validacao": base_validacao,
        "teste": base_teste,
    }.items()
])

display(auditoria_bases)

,conjunto,linhas,data_inicial,data_final,taxa_positivos
0,treino,16016,2023-03-28,2025-03-24,45.3546
1,validacao,3872,2025-04-01,2025-09-23,52.1436
2,teste,3784,2025-10-01,2026-03-21,51.7178
